# Puncta counter

__How to acquire the images:__
- This program works best with z-stacks of yeast cells
    - It is written for z-stacks where all wavelengths were first acquired for one z-position
    - For optimal cell counting, use 10 steps. You can then only use a subset of steps for the puncta counting.
- Acquire:
    - Images of the fluorescently labelled protein of interest
    - Images of a fluorescently labelled ER marker protein
    - Brightfield images

__How to set up the folder:__
- Set up a __master folder__ where all experiment folders to be analysed are put. Input the path to this folder as the "path" variable. This does not require change, as long as the same master folder is used.
    - Inside this folder, set up a folder for your __experiment__.
        - In this folder, the __results folder__ will appear as the program runs.
        - Inside the experiment folder, set up a folder for each __condition or strain__ of the experiment and split the images up accordingly.
            - This will be used to split up the images and get mean values for each condition.
            
__How to run the code:__
- Run all of the cells to read all functions
- Fill in the user input in the very bottom cell and press "Run interact" button

__This code will:__
- Import all images from the selected folder
- Create a max-z projection of the channel of the clustering protein
- Select the area of the cells present in the image and count the cells
- Detect protein clusters within the cell area and count them
    - Based on brightness, size and circularity of the clusters
    - All parameters are now designed to work best for the Snf7 puncta
- Plot different parameters in preliminary plots and export cell and cluster masks as tiff files for you to examine closer (for example in Fiji)

__The results folder:__
- Will be generated automatically in the __experiment__ folder.
- Will contain a subfolder for each condition, where the tiff files of the __cell mask__ and __puncta mask__ will be generated.
- Will contain the __preliminary graphs__ comparing the different conditions in different parameters
    - "...cells_with_puncta" = Bar graph of the number of the percentage of cells that contain puncta
    - "...intensities" = Bar graph of the puncta intensity per pixel
    - "...violin" = Violin plot showcasing the distribution of number of clusters inside a cell
    - "...puncta" = Bar graph of number of puncta per number of cells
- Will contain an __excel sheet__ with all the results
    - Comparison tab = Will contain all the mean values for each category
    - puncta_in_cells tab = Will contain all the numbers of clusters for each cell in each category
    - parameters tab = Will contain all the parameters that were selected by the user, so you can always check how this set of values was obtained.


## Upload data
- Here, each image set (z-stacks from all the channels) will be uploaded as NumPy arrays and put into a dictionary
- This version of the cluster counter is made only for tiff files

In [1]:
def upload_image(path, file): 
    
    # Create a complete path for the file
    folder_path = join(path, file)
    
    # Load the multipage tiff into python as a numpy array and get the number of pages in the tiff
    raw_img = imread(folder_path)
    pages, length, width = raw_img.shape
    
    # Create an empty dictionary in which the separated images will be stored
    img_dict = {}
     
    # For each page of the tiff, separated image will be stored as 2D numpy array
    for i in range(pages):
        img_dict['img' + str(i+1)] = raw_img[i]
 
    locals().update(img_dict)
    
    return(raw_img, img_dict)

#raw, img_dict = upload_image(os.path.join(param['path'], param['folder']), r'WT_Tm.tif')
#print(img_dict)

## Max intensity Z projection
- Max z projection will be created from all slices in each z-stack
- Here we look at a given pixel in each slice of the z-stack and only select the most intense one
- This will only be done for the protein of interest

In [2]:
def max_z(img_dict, param):
    
    # Create empty lists where only the slices belonging to the protein of interest will be stored
    Snf7_img = []
    final = []
    # Create a variable for the place the protein of interest has in the order of the z-stack
    num = param['Snf7']
    
    # Go through the pages of the generated dictionary
        # Starting with the page number containing the first image of our protein of interest
        # Keep increasing the number by 3 until you get to the end of the dictionary of images
        # Add all images of the protein of interest into the empty list
    while num <= len(img_dict.values()):
        for key,img in img_dict.items():
            if str(num) in key:
                Snf7_img.append(img)
                num +=3
    
    # Stack all the 2D numpy arrays into a 3D array
    z_stack = np.dstack(Snf7_img)      
    # This rotates the stacks, so here we turn the images back
    final_stack = np.rollaxis(z_stack,-1)
    # Make a max z projection of this 3D numpy array - we again end up with 2D numpy array
    z_max = np.max(z_stack, axis=2) 
    
    #print(z_max.shape)
    
    return(z_max)

## Convert to 8-bit
- The original image is a 16-bit image. This can cause the program to run a lot slower and give warnings.
- Therefore, the images are converted into 8-bit images while conserving the range of values as much as possible.

In [3]:
def convert8(im):
    
    # Now the value of each pixel is written as integers (whole numbers), they will be rewritten as floats  
    img = im.astype(dtype = "float32")
    
    # Image is scaled when converted into 8-bit image so that values brighter then 255 don´t all get labeled as 255
        # this way, the range of values will be preserved.
    im_temp = img - img.min()
    im_temp = im_temp / im_temp.max()
    im_temp = im_temp * 255

    im_uint8_scaled = im_temp.astype(np.uint8)
    mean = im_uint8_scaled.mean()
    
    return(im_uint8_scaled, mean)

## Count the cells (BF)
- Cell masks are created using a combination of intensity thresholding and size and circularity parameters
- This is done with a cortical section of the cells (one of the first or last slices of the brightfield z-stack)

In [4]:
def cell_count(image, mean, param, pr, zoom):
        
    # Gausian smoothing - smooths the image to minimize random gaps in the mask
    smooth_img = ndi.gaussian_filter(image, 5)
    
    # Adaptive thresholding
        # Background image is created where each pixel corresponds to a mean value of its surroundings.
        # The size of the surroundings is determined by the number 'i'
        # Binary image is generated where pixels !less! bright then their surroundings are set as 1
    i = 113
    struct = (np.mgrid[:i,:i][0] - np.floor(i/2))**2 + (np.mgrid[:i,:i][1] - np.floor(i/2))**2 <= np.floor(i/2)**2
    from skimage.filters.rank import mean
    background = mean(smooth_img, footprint=struct)
    rough_img = smooth_img < background
    
    # Erosion
        # We remove a certain number of pixels from the outside of each structure.
        # This will allow us to separate small protrusions that are not actually part of the thick cell border
    eroded_img = ndi.binary_erosion(rough_img, iterations = 1)
    
    # Remove small objects - removes small noise that got picked up in thresholding
    removed_img = remove_small_objects(eroded_img, min_size = 1000)
            
    # Padding and closing - This will connect nearby structures to properly close the border of a cell
    j = 41

    SE = (np.mgrid[:j,:j][0] - np.floor(j/2))**2 + (np.mgrid[:j,:j][1] - np.floor(j/2))**2 <= np.floor(j/2)**2
    pad_size = j+1
    padded_img = np.pad(removed_img, pad_size, mode='reflect')
    closed_img = ndi.binary_closing(padded_img, structure=SE)
    improved_img = closed_img[pad_size:-pad_size, pad_size:-pad_size]
    
    # As we have now created a mask containing the borders of each cell, we now need to invert the mask
    inv_img = ~improved_img
    
    # Label every separated cluster of 1s
        # Each cluster of 1s will get assigned a different number (we will get a cluster of 2s, 3s etc.)
    preliminary, num_pre = ndi.label(inv_img)
    # A lot of the background is removed based on the size of the object
    for i in np.unique(preliminary):
        obj = preliminary==i
        if np.sum(obj) > 10000:
            preliminary[obj] = 0
    
    mask1 = np.array(preliminary, dtype = bool)
    
    # To try to separate cells that might have been picked up as one structure
    # We select pixels most distant from the background
    mask_dt = ndi.distance_transform_edt(mask1)
    mask_dt_smooth = ndi.gaussian_filter(mask_dt, 3)
    from skimage.feature import peak_local_max
    local_max_coords = peak_local_max(mask_dt_smooth, min_distance = 25)
    local_max_mask = np.zeros_like(mask_dt_smooth, dtype=bool)
    local_max_mask[tuple(local_max_coords.T)] = True
    peaks, num_peaks = ndi.label(local_max_mask)
    # We expand the peak pixels with watershed (It will expand until it meets another expanding structure)
    from skimage.segmentation import watershed
    mask_dt_inv = np.logical_not(mask_dt_smooth)
    mask_expanded = watershed(mask_dt_inv, peaks, mask = mask1)
    
    # Set up some empty lists and arrays of zeros to fill in later
    final = np.zeros_like(mask_expanded)
    label_num = 0
    percentages = []
    perimeters = []
    
    # Create a mask which includes the border pixels to filter out cells which are touching the border
    border_mask = np.ones_like(mask_expanded)
    border_mask[2:-2, 2:-2] = 0
    
    # Itterate through the structures and filter them out based on different parameters
    for label in np.unique(mask_expanded[mask_expanded != 0]):
        mask_im = mask_expanded == label
        
        # Remove cells that touch the border
        cells_and_border = np.logical_and(mask_im, border_mask)
        if True in cells_and_border:
            pass
        
        # Remove objects which are too big or too small
        elif 10000 < np.sum(mask_im) or np.sum(mask_im) < 1500:
            pass
        
        else:
            # Calculate the perimeter of the cell
            mask_eroded = ndi.binary_erosion(mask_im)
            perimeter_cell = np.sum(np.logical_xor(mask_im, mask_eroded))

            # Label the middle of the cell based on peaks which were used for watershed
            peak = peaks == label
            # Calculate the radius of a perfect circle with the same area
            r = math.sqrt(np.sum(mask_im)/math.pi)
            # Create a circle with the same radius with a middle located at the coordinates of our object's middle
            nika = expand_labels(peak, distance=r)
            # Calculate the perimeter of this perfect circle
            nika_eroded = ndi.binary_erosion(nika)
            perimeter_nika = np.sum(np.logical_xor(nika, nika_eroded))
            
            # Compare the perimeter of our cell and the perimeter of the perfect circle with same area
            perimeter_comp = (perimeter_nika/perimeter_cell)*100
            # Compare the shape of our cell and the perfect circle with the same area
            compare = np.logical_and(nika, mask_im)
            if np.sum(compare) == 0:
                percentage = 0
            else:
                percentage = (np.sum(compare)/np.sum(nika))*100
            
            # Collect the info about all the objects so you can theoretically look at it to set the param etc.
            # this is not being exported yet
            percentages.append(percentage)
            perimeters.append(perimeter_comp)
            
            # Remove cells which don't correspond to the circularity parameters
            if percentage > 85 and perimeter_comp > 90:
                label_num += 1
                final[mask_expanded==label] = label_num
    
    # Count the final number of cells
    count = len(np.unique(final[final != 0]))
    # Create a binary cell mask
    cells_mask = final.astype(dtype = bool)
    # Export the cell mask as a tiff file
    imwrite(join(param['results'], param['category'], f'{param["im"]}_cell_mask.tif'), final)
    # Print the number of cells to monitor the progress of the program
    print(f'Number of cells in {param["im"]}: {count}')
    
    # Vizualise the results
    if pr == 'yes':
        if zoom == 'none':
            f, axarr = plt.subplots(4,2, figsize=(10,20))
            axarr[0,0].imshow(smooth_img, cmap='gray', interpolation='none')
            axarr[0,0].set_title('Smooth')
            axarr[0,1].imshow(rough_img, cmap='gray', interpolation='none')
            axarr[0,1].set_title('Adaptive thresholding')
            axarr[1,0].imshow(eroded_img, cmap='gray', interpolation='none')
            axarr[1,0].set_title('Erosion')
            axarr[1,1].imshow(removed_img, cmap='gray', interpolation='none')
            axarr[1,1].set_title('Remove small objects')
            axarr[2,0].imshow(improved_img, cmap='gray', interpolation='none')
            axarr[2,0].set_title('Padding and closing')
            axarr[2,1].imshow(preliminary, cmap='gray', interpolation='none')
            axarr[2,1].set_title(f'preliminary labeling counts {num_pre} objects')
            axarr[3,0].imshow(image, cmap='gray', interpolation='none')
            axarr[3,0].imshow(np.ma.array(mask_expanded,mask = mask_expanded==0),
                              cmap='prism', interpolation='none')
            axarr[3,0].set_title('Watershed')
            axarr[3,1].imshow(image, cmap='gray', interpolation='none')
            axarr[3,1].imshow(np.ma.array(final, mask = final==0),
                              cmap='prism', alpha = 0.4, interpolation='none')
            axarr[3,1].set_title(f'Final number of cells is {count}')
            
        else:
            zoom = param['zoom']
            
            f, axarr = plt.subplots(4,2, figsize=(10,20))
            axarr[0,0].imshow(smooth_img[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            axarr[0,0].set_title('Smooth')
            axarr[0,1].imshow(rough_img[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            axarr[0,1].set_title('Adaptive thresholding')
            axarr[1,0].imshow(eroded_img[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            axarr[1,0].set_title('Erosion')
            axarr[1,1].imshow(removed_img[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            axarr[1,1].set_title('Remove small objects')
            axarr[2,0].imshow(improved_img[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            axarr[2,0].set_title('Padding and closing')
            axarr[2,1].imshow(preliminary[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            axarr[2,1].set_title(f'preliminary labeling counts {num_pre} objects')
            axarr[3,0].imshow(image[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            axarr[3,0].imshow(np.ma.array(mask_expanded[zoom[0]:zoom[1], zoom[2]:zoom[3]],
                                          mask = mask_expanded[zoom[0]:zoom[1], zoom[2]:zoom[3]]==0),
                              cmap='prism', interpolation='none')
            axarr[3,0].set_title('Watershed')
            axarr[3,1].imshow(image[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            axarr[3,1].imshow(np.ma.array(final[zoom[0]:zoom[1], zoom[2]:zoom[3]],
                                          mask = final[zoom[0]:zoom[1], zoom[2]:zoom[3]]==0),
                              cmap='prism', alpha = 0.4, interpolation='none')
            axarr[3,1].set_title(f'Final number of cells is {count}')
            
        plt.figure(figsize=(10,10))
        plt.imshow(image, cmap='gray', interpolation='none')
        plt.imshow(np.ma.array(final, mask = final==0), cmap='prism', alpha = 0.4, interpolation='none')
        plt.title(f'Cells: {len(np.unique(final[final != 0]))}')
    
    return(count, final, cells_mask)

## Find potential puncta
- First, we create a Boolean mask of potential puncta based on intensity thresholding
- Then we label them and filter them based on their size and circularity

In [5]:
def image_proces(name, image, cells, mean_intensity, param, pr, zoom):
    
    # Adaptive thresholding
        # Background image is created where each pixel corresponds to a mean value of its surroundings.
        # The size of the surroundings is determined by the number 'i' in parameters
        # Binary image is generated where pixels brighter then their surroundings are set as 1
    j = param['i']
    struct = (np.mgrid[:j,:j][0] - np.floor(j/2))**2 + (np.mgrid[:j,:j][1] - np.floor(j/2))**2 <= np.floor(j/2)**2
    background = mean(image, footprint=struct)
    mask = image > param['times']*background
    
    # Now we filter out background based on our cell mask
        # Everything outside of a slightly expanded cell mask is set as 0
    cells_expanded = expand_labels(cells, distance = 5)
    mask[cells_expanded == 0] = 0
    
    # Label every separated cluster of 1s
        # Each cluster of 1s will get assigned a different number (we will get a cluster of 2s, 3s etc.)
    preliminary, num = ndi.label(mask)
            
    # Here we set up some empty lists and arrays of zeros to fill in later
    percentages = []
    perimeters = []
    final = np.zeros_like(mask, dtype = 'uint16')
    label_num = 0
    
    # Asses the circularity and size of each labeled object
    for label in np.unique(preliminary[preliminary != 0]):
        mask_im = preliminary == label
        
        # Get rid of objects which are extremely big
        if param['Max_size'] < np.sum(mask_im) or np.sum(mask_im) < param['Min_size']:
            pass
        
        else:
            # Calculate the perimeter of the object
            mask_eroded = ndi.binary_erosion(mask_im)
            perimeter_object = np.sum(np.logical_xor(mask_im, mask_eroded))
        
            # Label the middles of the object by detecting the pixel which is the furthest from the background
            mask_dt = ndi.distance_transform_edt(mask_im)
            mask_dt_smooth = ndi.gaussian_filter(mask_dt, 5)
            local_max_coords = peak_local_max(mask_dt_smooth, min_distance=60)
            local_max_mask = np.zeros_like(mask_dt_smooth, dtype=bool)
            local_max_mask[tuple(local_max_coords.T)] = True
            peak, num_peaks = ndi.label(local_max_mask)
            
            # Calculate the radius of a perfect circle with the same area as our object
            r = math.sqrt(np.sum(mask_im)/math.pi)
            # Create a circle with the same radius with a middle located at the coordinates of our object's middle
            nika = expand_labels(peak, distance=r)
            # Calculate the perimeter of the perfect circle
            nika_eroded = ndi.binary_erosion(nika)
            perimeter_nika = np.sum(np.logical_xor(nika, nika_eroded))
            
            # Compare the perimeters of our objects and of the perfect circle with the same area
            perimeter_comp = (perimeter_nika/perimeter_object)*100
            # Compare the shape of our object and the perfect circle with the same area
            compare = np.logical_and(nika, mask_im)
            if np.sum(compare) == 0:
                percentage = 0
            else:
                percentage = (np.sum(compare)/np.sum(nika))*100
            
            # Collect the info about all the objects so you can theoretically look at it to set the param etc.
            # this is not being exported yet
            percentages.append(percentage)
            perimeters.append(perimeter_comp)
            
            # If the object corresponds to both circularity parameters, it gets added to our final mask
            if percentage > param['areaC'] and perimeter_comp > param['perimeterC']:
                label_num += 1
                final[preliminary==label] = label_num
    
    # Calculate the final number of puncta
    puncta_num = len(np.unique(final[final != 0]))
    # Create a boolean array from the labeled mask
    puncta_mask = np.array(final, dtype = bool)
    
    # Create a mask where the puncta are circled for better visualization in Fiji and save it as a tiff
    puncta_edges = np.zeros_like(final, dtype = 'uint16')
    expanded = expand_labels(final, distance=10)
    for puncta_ID in np.unique(expanded[expanded != 0]):
        p_mask = expanded==puncta_ID
        eroded_p_mask = ndi.binary_erosion(p_mask, iterations=5)
        edge_mask = np.logical_xor(p_mask, eroded_p_mask)
        puncta_edges[edge_mask] = puncta_ID
        
    imwrite(join(param['results'], param['category'], f'{param["im"]}_puncta_mask.tif'), puncta_edges)
    imwrite(join(param['results'], param['category'], f'{param["im"]}_image.tif'), image)
    
    
    # Show the images, to see the progress of puncta detection
    if pr == 'yes':
        if zoom=='none':
            f, axarr = plt.subplots(2,2, figsize=(10,10), sharey=True)
            axarr[0,0].imshow(image, interpolation="none", cmap="gray")
            axarr[0,0].set_title(f'Image')
        
            axarr[0,1].imshow(mask, interpolation="none", cmap="gray")
            axarr[0,1].set_title(f'Adaptive thresholding')
        
            axarr[1,0].imshow(image, interpolation="none", cmap="gray")
            axarr[1,0].imshow((np.ma.masked_where(preliminary == 0, preliminary)),
                              interpolation = "none", cmap="prism")
            axarr[1,0].set_title(f'Before size and circularity filtering')
                    
            axarr[1,1].imshow(image, cmap="gray", interpolation="none")
            axarr[1,1].imshow((np.ma.masked_where(puncta_edges == 0, puncta_edges)),
                              interpolation = "none", cmap="prism")
            axarr[1,1].set_title(f'{puncta_num} potential puncta in {name}')
                        
        else:
            zoom = param['zoom']
            f, axarr = plt.subplots(2,2, figsize=(5,20), sharey=True)
            axarr[0,0].imshow(image[zoom[0]:zoom[1], zoom[2]:zoom[3]], interpolation="none", cmap="gray")
            axarr[0,0].set_title(f'Image')
                
            axarr[0,1].imshow(mask[zoom[0]:zoom[1], zoom[2]:zoom[3]], interpolation="none", cmap="gray")
            axarr[0,1].set_title(f'Adaptive thresholding')
        
            axarr[1,0].imshow(image[zoom[0]:zoom[1], zoom[2]:zoom[3]], interpolation="none", cmap="gray")
            axarr[1,0].imshow((np.ma.masked_where(preliminary[zoom[0]:zoom[1], zoom[2]:zoom[3]] == 0,
                                                  preliminary[zoom[0]:zoom[1], zoom[2]:zoom[3]])),
                              interpolation = "none", cmap="prism")
            axarr[1,0].set_title(f'Labeling 1')
                    
            axarr[1,1].imshow(image[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap="gray", interpolation="none")
            axarr[1,1].imshow((np.ma.masked_where(puncta_edges[zoom[0]:zoom[1], zoom[2]:zoom[3]] == 0,
                                                  puncta_edges[zoom[0]:zoom[1], zoom[2]:zoom[3]])),
                              interpolation = "none", cmap="prism")
            axarr[1,1].set_title(f'{puncta_num} potential puncta in {name}')
        
        plt.figure(figsize=(10,10))
        plt.imshow(image, cmap='gray', interpolation='none')
        plt.imshow(np.ma.array(puncta_edges, mask = puncta_edges==0), cmap='prism', alpha = 0.4, interpolation='none')
        plt.title(f'Final puncta: {len(np.unique(final[final != 0]))}')
                
    return(final, puncta_num, puncta_mask)

## Number of puncta in each cell
- Each puncta label is assigned to a cell label to know how many clusters are typically in one cell
- Number of clusters for each cell is recorded as a dictionary

In [6]:
def puncta_in_cell(image, cells, puncta, cell_count, param, pr, zoom):
    
    # Expand the cell labels a little bit to make sure we are including all of the puncta inside of them.
        # Sometimes the cytoplasmic area detected is a couple pixels smaller then the acctual cell
        # The labels will stop expanding when they touch another label, so there will be no overlap
    expanded_cells = expand_labels(cells, distance=5)
    puncta_final = np.copy(puncta)
    puncta_in = np.zeros_like(puncta_final)
    
    # Create a dictionary where each cell label is a key, ready to be assigned a value:
    puncta_in_cell = dict.fromkeys(np.unique(cells[cells != 0]), 0)
    
    # Itterate over all of the puncta labels
    for i in np.unique(puncta[puncta != 0]):
        # If the punctum is inside of a cell, it will be marked with the same number as that cell
        if 0 not in expanded_cells[puncta == i]:
            puncta_in[puncta == i] = expanded_cells[puncta == i]
            cell_number = np.unique(expanded_cells[puncta == i])[0]
            puncta_in_cell[cell_number] += 1
        # Othervise it will be removed from the final puncta mask
        else:
            puncta_final[puncta_final == i] = 0
    
    # Count the cells with and without puncta
    cells_without = sum(x == 0 for x in puncta_in_cell.values())
    cells_with = sum(y != 0 for y in puncta_in_cell.values())
    # Get a percentage of cells with puncta
    cells_with_puncta = (cells_with/cell_count)*100
    
    # Print the final number of clusters to monitor the process
    print(f'Number of puncta in {param["im"]}: {len(np.unique(puncta_final[puncta_final != 0]))}')
    
    # Create a mask where the puncta are circled for better visualization
    puncta_edges = np.zeros_like(puncta_final, dtype='uint8')
    expanded = expand_labels(puncta_final, distance=8)
    expanded_m = np.array(expanded, dtype = bool)
    for puncta_ID in np.unique(expanded[expanded != 0]):
        p_mask = expanded==puncta_ID
        eroded_p_mask = ndi.binary_erosion(p_mask, iterations=3)
        edge_mask = np.logical_xor(p_mask, eroded_p_mask)
        puncta_edges[edge_mask] = puncta_ID
    
    # Visualize the results
    if pr == 'yes':
        if zoom == 'none':
            plt.figure(figsize=(10,10))
            plt.imshow(expanded_cells, cmap='gray', interpolation='none')
            plt.imshow(np.ma.array(puncta_final, mask = puncta_final==0), cmap='prism', interpolation='none')
            plt.title(f'Cells with puncta: {cells_with_puncta:.2f}%')
        
            plt.figure(figsize=(10,10))
            plt.imshow(image, cmap='gray', interpolation='none')
            plt.imshow(np.ma.array(puncta_edges, mask = puncta_edges==0), cmap='prism', interpolation='none')
            plt.title(f'Final puncta count: {len(np.unique(puncta_final[puncta_final != 0]))}%')
            
        else:
            zoom = param['zoom']
            plt.figure(figsize=(10,10))
            plt.imshow(image[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            plt.imshow(np.ma.array(expanded_cells[zoom[0]:zoom[1], zoom[2]:zoom[3]],
                                   mask = expanded_cells[zoom[0]:zoom[1], zoom[2]:zoom[3]]==0),
                                   alpha=0.3, cmap='prism', interpolation='none')
            plt.imshow(np.ma.array(puncta_final[zoom[0]:zoom[1], zoom[2]:zoom[3]],
                                   mask = puncta_final[zoom[0]:zoom[1], zoom[2]:zoom[3]]==0),
                                   cmap='prism', interpolation='none')
            plt.title(f'Cells with puncta: {cells_with_puncta:.2f}%')
            
            plt.figure(figsize=(10,10))
            plt.imshow(image[zoom[0]:zoom[1], zoom[2]:zoom[3]], cmap='gray', interpolation='none')
            plt.imshow(np.ma.array(puncta_edges[zoom[0]:zoom[1], zoom[2]:zoom[3]],
                                   mask = puncta_edges[zoom[0]:zoom[1], zoom[2]:zoom[3]]==0),
                                   cmap='prism', interpolation='none')
            plt.title(f'Final puncta count: {len(np.unique(puncta_final[puncta_final != 0]))}')
                                    
    return(puncta_in_cell, puncta_in, puncta_final, puncta_edges, cells_with_puncta)

## Puncta intensities
- Calculate the intensity of the clusters, normalised to the intensity of signal of the whole cell mask

In [7]:
def puncta_intensity(img, puncta, cells):
    
    # Get mean brightness of the area where cells are located
    cell_b = np.mean(img[cells])
    
    brightness = []
    
    # Calculate the brightness for each cluster and normalize it to the whole-cell-area brightness
    for i in np.unique(puncta[puncta != 0]):
        mask = puncta==i
        b = np.mean(img[mask==1])/cell_b
        brightness.append(b)
    
    # To visualy separate images to make it easier to follow the progress of the program
    print('_____________________________________________________________________')
    
    return(brightness)

## Perform all the functions for one field of view
- This function receives all the images for one field of view as a path to the folder + list of file names

In [8]:
def ZnamkaPunctu(path, files, param, pri, zoom):
    
    # Change the 'im' parameter to the current one
    param['im'] = files
    
    # Upload the images as a library of 2D numpy arrays
    raw, img_lib = upload_image(path, files)
    
    # Get dimensions of the image and number of images, mostly as a control
    pages, length, width = raw.shape
    
    # Max Z projection of the channel of the protein of interest
    Snf7_max = max_z(img_lib, param)
    
    # Convert to 8-bit integer arrays
    Snf7, Snf7_mean = convert8(Snf7_max)
    ER, ER_mean = convert8(img_lib[f'img{param["ER"]}'])
    BF, BF_mean = convert8(img_lib[f'img3'])
    
    # Count the cells in this field of view using the brightfield image
    cell_num, cells_label, cells_mask = cell_count(BF, BF_mean, param, pri, zoom)
    
    # Label potential puncta
    Snf7_label, Snf7_number, Snf7_mask = image_proces('Snf7', Snf7, cells_mask, Snf7_mean, param, pri, zoom)
    
    # Count puncta in each cell
    pinc_dict, pinc_img, puncta_final, edges, c_with_p_percentage = puncta_in_cell(Snf7, cells_label, Snf7_label, 
                                                                                   cell_num, param, pri, zoom)   
    # Calculate the final number of puncta per number of cells
    count_final = len(np.unique(puncta_final[puncta_final != 0]))
    if cell_num != 0:
        pc = count_final/cell_num
    else:
        print('No cells counted')
        pc = 0
    
    # Compare the intensity of the puncta to the overall signal in the cell
    intensity = puncta_intensity(Snf7, cells_mask, puncta_final)
    
    # All relevant numpy array and number values get stored in a dictionary, which is the output of this function
        # The number values will then get combined with all fileds of view from the same condition / strain
    images ={'img': raw,
             'Snf7_max': Snf7_max,
             'Snf7_puncta': Snf7_label,
             'cell_m': cells_label,
             'puncta_edges': edges,
             'puncta_into_cells': pinc_img}
    
    numbers = {'puncta_n': count_final,
               'cell_n': cell_num,
               'puncta_per_cell': pc,
               'cells_percentage': c_with_p_percentage,
               'puncta_in_cell': pinc_dict.values(),
               'intensity': intensity}
                    
    return(images, numbers)

## Extracting mean values for each category
- This function runs for one experimental category (one treatment condition or perhaps one strain)
- Here we take images belonging to one category, split them into each field of view and analyse them one by one
- In the end, we will end up with a dictionary of values of each field of view in this category

In [9]:
def category_mean(category_path, param):
    
    # Get all of the image names in the given category folder
    images = os.listdir(category_path)
    print(images)
    # Create a subfolder in the results folder, where the tiff files of cell and puncta masks will be exported
    os.makedirs(join(param['results'], param['category']))
    
    # Create an empty library with keys corresponding to the values to be collected for each field of view
    final ={'frame':[],
            'puncta_num':[],
            'cell_num':[],
            'puncta_per_cell':[],
            'cells_percentage':[],
            'puncta_in_cell':[],
            'intensity':[]}
    
    # Here we identify each tiff file in the category folder
    for image in glob.glob(join(category_path, '*.tif'), recursive = True):
        im = image.split("/")[-1]
        
        # And perform all image analysis steps for each tiff file
        images, numbers = ZnamkaPunctu(category_path, im, param, 'no', 'none')
        
        # Values from each field of view are added into this dictionary, to be compared with other categories
        final['frame'].append(im)
        final['puncta_num'].append(numbers['puncta_n'])
        final['cell_num'].append(numbers['cell_n'])
        final['puncta_per_cell'].append(numbers['puncta_per_cell'])
        final['cells_percentage'].append(numbers['cells_percentage'])
        final['puncta_in_cell'].extend(numbers['puncta_in_cell'])
        final['intensity'].extend(numbers['intensity'])
        
        # Visualize the masks for each field of view to monitor the running of the pipeline
        if param['show_imgs'] == True:
            f, axarr = plt.subplots(1,3, figsize=(10,14), sharey=True)
            axarr[0].imshow(images['img'][2], cmap='gray', interpolation='none')
            axarr[0].imshow(np.ma.array(images['cell_m'], mask = images['cell_m']==0), cmap='prism',
                            alpha = 0.5, interpolation='none')
            axarr[0].set_title(f"Puncta in Snf7: {len(np.unique(images['Snf7_puncta']))-1}")
            axarr[1].imshow(images['Snf7_max'], cmap='gray', interpolation='none')
            axarr[1].imshow(np.ma.array(images['puncta_edges'], mask = images['puncta_edges']==0), cmap='prism',
                            alpha = 1, interpolation='none')
            axarr[1].set_title(f'Puncta in {im}: {numbers["puncta_n"]}')
            axarr[2].imshow(images['Snf7_max'], cmap='gray', interpolation='none')
            axarr[2].imshow(np.ma.array(images['cell_m'], mask = images['cell_m']==0), cmap='winter', alpha = 0.4,
                            interpolation='none')
            axarr[2].imshow(np.ma.array(images['Snf7_puncta'], mask = images['Snf7_puncta']==0), cmap='autumn',
                            alpha = 1, interpolation='none')
            axarr[2].set_title(f'There are on average {numbers["cells_percentage"]:.2} puncta in a cell')
        
    # Plot number of puncta per cell for each frame in the category
    x_axis1 = final['frame']
    y_axis1 = final['puncta_per_cell']

    plt.figure(figsize=(5,3))
    plt.figure(figsize=(5,3))
    plt.bar(x_axis1, y_axis1)
    plt.title(f'Number of puncta per cell in {category_path.split("/")[-1]}')
    plt.xlabel('Sample number')
    plt.ylabel('Puncta per cell')
    plt.show()
        
    # Plot percentage of cells with puncta for each frame in the category
    x_axis2 = final['frame']
    y_axis2 = final['cells_percentage']

    plt.figure(figsize=(5,3))
    plt.figure(figsize=(5,3))
    plt.bar(x_axis2, y_axis2)
    plt.title(f'Percentage of cells with puncta in {category_path.split("/")[-1]}')
    plt.xlabel('Sample number')
    plt.ylabel('Percentage of cells with puncta')
    plt.show()
    
    return(final)

## Comparison of each category
- The program looks into the experiment folder and runs the rest of the functions for each experiment category
- In the end we get mean values for each experiment category
- Preliminary graphs will be plotted to quickly showcase the results
- All mean values will be exported in an Excel sheet, along with the parameters used for this analysis

In [10]:
def batch(param):
    # To measure time it takes to compute everything
    from datetime import datetime
    startTime = datetime.now()
    
    # Get the path to our main folder
    folder_path = join(param['path'], param['folder'])
    # Look inside the folder to see what category folders are inside and create a list of these
    data_categories = os.listdir(folder_path)
    # This is removed to avid confusion
    if '.DS_Store' in data_categories:
        data_categories.remove('.DS_Store')
    # If the categories are numerical (time of treatment / conc of reagent etc.), they are sorted by number
    if data_categories[0].isdigit():
        data_categories.sort(key=int)
    
    print(data_categories)
    
    # Create a library where the mean values for each category will be added
    comparison ={'category':[],
                 'puncta':[],
                 'cells':[],
                 'puncta_per_cell':[],
                 'cells_with_puncta':[],
                 'intensity':[],
                 'max_intensity':[],
                 'min_intensity':[]}
    
    # Create an empty dictionary where the number of puncta in each cell will be added
    p_in_c = {}
    
    # For each category, perform the analysis and obtain the mean values
    for category in data_categories:
        if "_counted_" in category:
            pass
        else:
            category_path = join(folder_path, category)
            param['category'] = category
            final = category_mean(category_path, param)
        
            comparison['category'].append(category)
            comparison['puncta'].append(final['puncta_num'])
            comparison['cells'].append(final['cell_num'])
            comparison['puncta_per_cell'].append(np.mean(final['puncta_per_cell']))
            comparison['cells_with_puncta'].append(np.mean(final['cells_percentage']))
            comparison['intensity'].append(np.mean(final['intensity']))
            comparison['max_intensity'].append(max(final['intensity']))
            comparison['min_intensity'].append(min(final['intensity']))
        
            p_in_c[category] = final['puncta_in_cell']
            
    # Creating pandas data frames which will be used to create our graphs and generate the Excel sheet
    df = pd.DataFrame(data=comparison)
    
    parameters = pd.DataFrame(data=param, index = [0])
    
    pinc = pd.DataFrame.from_dict(p_in_c, orient='index')
    pf = pinc.transpose()
    
    # Here we generate an Excel sheet containing:
        # All the mean values for each category
        # Puncta numbers in each cell
        # Parameters used for this analysis
    with pd.ExcelWriter(os.path.join(param['results'],f'{param["folder"]}.xlsx')) as writer:
        df.to_excel(writer, sheet_name='comparison')
        parameters.to_excel(writer, sheet_name='parameters')
        pinc.to_excel(writer, sheet_name='puncta_in_cells')
        
    # Violin plot of number of puncta in one cell
    sns.violinplot(data = pf, inner='point', density_norm='area')
    plt.title(f'Number of puncta in one cell')
    plt.savefig(os.path.join(param['results'], f'{param["folder"]}_violin.pdf'))
    plt.show()
    
    # Number of puncta per cell
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(x='category', y='puncta_per_cell', data=df, ax=ax)
    plt.title(f'Comparison of puncta per cell in {param["folder"]}')
    plt.xlabel('Condition')
    plt.ylabel('Mean puncta per cell')
    plt.savefig(os.path.join(param['results'], f'{param["folder"]}_puncta.pdf'))
    plt.show()
    
    # Percentage of cells with puncta
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(x='category', y='cells_with_puncta', data=df, ax=ax)
    plt.title(f'Comparison of percentage of cells with puncta in {param["folder"]}')
    plt.savefig(os.path.join(param['results'], f'{param["folder"]}_cells_with_puncta.pdf'))
    plt.show()
           
    # Plot puncta intensities
    plt.rcParams['errorbar.capsize'] = 10
    
    # create ymin and ymax
    df['yminI'] = df.intensity - df.min_intensity
    df['ymaxI'] = df.max_intensity - df.intensity

    # extract ymin and ymax into a (2, N) array as required by the yerr parameter
    yerrI = df[['yminI', 'ymaxI']].T.to_numpy()

    # plot with error bars
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(x='category', y='intensity', data=df, yerr=yerrI, ax=ax)
    plt.title(f'Comparison of puncta intensities in {param["folder"]}')
    plt.savefig(os.path.join(param['results'],f'{param["folder"]}_intensities.pdf'))
    plt.show()
     
    # Print out how long it took to run the entire pipeline for this folder
    print(datetime.now() - startTime)
    
    return(df, pinc)

## Input parameters
- Run this cell to get the interface for imputing parameters
- Make sure that you have run all of the previous cells so all the functions are ready

In [11]:
# Import all the modules needed
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
import pandas
from os.path import join
from tifffile import imread
from tifffile import imwrite
import math
from skimage import data
from skimage.morphology import remove_small_objects
from skimage.segmentation import expand_labels
import skimage
from skimage.filters.rank import mean
from skimage.feature import peak_local_max
import glob
import os
import pandas as pd
import seaborn as sns

from IPython.display import display
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

def parameters(Folder_name, Snf7, ER, Adapt_times, Adapt_size, Size, Perimeter, Area, Show_imgs):
    # Creating new folder which will contain our results.
    # If this folder already exists, it will automatically increase the number at the end.
    
    # WRITE THE PATH TO THE GENERAL FOLDER WITH ALL EXPERIMENT FOLDERS TO BE ANALYSED
    path = r'/...'
    # Path to your choosen folder is created
    folder_path = os.path.join(path, Folder_name)
    
    n = 1
    ex = 'yes'
    
    # Create a results folder.
    # If results folder of a given number is already present, it will create a new folder with higher number.
    # So the results will not be overwritten if you run this code multiple times for the same experiment.
    while ex == 'yes':
        new_folder = f'{Folder_name}_puncta_counted_{n}'
        new_path = os.path.join(folder_path, new_folder)
        if os.path.exists(new_path):
            n += 1
        else:
            os.makedirs(new_path)
            ex = 'no'
    
    # Here we asign all the user imput parameters to a parameter dictionary
    param = {'path': path,
             'folder': Folder_name,
             'results': new_path,
             'times': float(Adapt_times),
             'i': float(Adapt_size),
             'Min_size': float(Size[0]),
             'Max_size': float(Size[1]),
             'perimeterC': float(Perimeter),
             'areaC': float(Area),
             'show_imgs': Show_imgs,
             'ER': ER,
             'Snf7': Snf7}
    
    # Now we run the batch function for the entire folder
    comparison, pinc = batch(param)
    return()

# Here the interface to input user data is defined
# By changing the "value=___" at the end of each line, you can change the default value
interact_manual(parameters, continuous_update = False, 
    Folder_name = '', # Name of the experiment folder
    
    # The order in which the image was recorded
    Snf7 = widgets.Dropdown(options=[(1), (2), (3), ('None')],value=2),
    ER = widgets.Dropdown(options=[(1), (2), (3),('None')],value=1),
                
    # Parameters for creating the cluster mask:
    Adapt_times = widgets.FloatSlider(min=0, max=20, step=0.5, value = 5), # Adaptive thresholding goal
    Adapt_size = '133', # Size of the area for adaptive thresholding
    Size = widgets.IntRangeSlider(min = 0, max = 800, step = 1, value = [10, 600]), # Range for size filtering (px)
                
    # Percentages for circularity filtering:
    Perimeter = '50', # Based on the perimeter calculation of a true circle with the same area
    Area = '70', # Based on overlap with the area of the true circle with the same middle as our cluster
                
    Show_imgs = True) # Do you want to be shown images of the masks to monitor the process

interactive(children=(Text(value='example images', continuous_update=False, description='Folder_name'), Dropdo…

<function __main__.parameters(Folder_name, Snf7, ER, Adapt_times, Adapt_size, Size, Perimeter, Area, Show_imgs)>